# 1. Setup Environment (Clone Repo & Install)
Jalankan sel ini pertama kali untuk mengunduh project dari GitHub dan menginstal dependensi.

In [ ]:
# CELL 0: Persiapan Environment
import os

# 1. Clone repositori GitHub jika belum ada di Colab
if not os.path.exists('/content/BlinkLink-Iot'):
    !git clone https://github.com/assidik12/BlinkLink-Iot.git

# 2. Pindah direktori kerja ke dalam folder project
%cd /content/BlinkLink-Iot

# 3. Install dependensi yang diperlukan (seperti keras-facenet, paho-mqtt, mediapipe)
!pip install -r requirements.txt

print("\n✅ Setup Selesai! Posisi direktori saat ini:")
!pwd

# 2. Open Camera Using JS from window
Menyiapkan jembatan JavaScript untuk mengakses webcam laptop dan mengirimkannya ke Python di Colab.

In [ ]:
# CELL 1: JavaScript Camera Bridge (Input System)
from IPython.display import display, HTML
from google.colab.output import eval_js
import base64
import cv2
import numpy as np

def init_camera():
    """Inisialisasi webcam menggunakan HTML/JS di environment browser Colab"""
    display(HTML('''
        <video id="videoElement" width="320" height="240" autoplay style="display: none;"></video>
        <canvas id="canvasElement" width="320" height="240" style="display: none;"></canvas>
        <script>
            async function setupCamera() {
                const video = document.getElementById('videoElement');
                const stream = await navigator.mediaDevices.getUserMedia({video: true});
                video.srcObject = stream;
                // Menyimpan referensi elemen untuk diakses oleh get_frame()
                window.colab_video = video;
                window.colab_canvas = document.getElementById('canvasElement');
            }
            setupCamera();
        </script>
    '''))

def get_frame():
    """Menangkap satu frame dari webcam JS dan merubahnya menjadi numpy array BGR"""
    try:
        data_url = eval_js('''
            (function() {
                if (!window.colab_video || !window.colab_canvas) return null;
                const ctx = window.colab_canvas.getContext('2d');
                ctx.drawImage(window.colab_video, 0, 0, 320, 240);
                // Kompresi JPEG sebesar 0.7 untuk kelancaran bandwidth I/O
                return window.colab_canvas.toDataURL('image/jpeg', 0.7);
            })();
        ''')
        
        if not data_url:
            return None
            
        # Mengubah string Base64 ke OpenCV Frame (Numpy Array)
        img_b64 = data_url.split(',')[1]
        img_bytes = base64.b64decode(img_b64)
        img_arr = np.frombuffer(img_bytes, dtype=np.uint8)
        img_cv = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        return img_cv
    except Exception as e:
        return None

print("✅ Fungsi jembatan kamera berhasil dimuat!")
print("ℹ️ Catatan: Popup izin kamera (Allow Camera) baru akan muncul saat Anda mulai menjalankan CELL 3.")


# 3. Initialize Models & Connect MQTT
Memuat model AI (FaceNet & MediaPipe) dari folder lokal project dan mengaktifkan klien IoT.

In [ ]:
# CELL 2: Imports & Initialization
import sys
import os
import cv2

# 1. Mencegah ALSA/Pygame Error di headless mode
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'

# 2. Import Module dari Project BlinkLink-Iot
try:
    from vision_controller.mp_face_detector import FaceMeshDetector
    from vision_controller.face_auth import FaceAuthenticator 
    from preprocessing.blinker import BlinkProcessor 
    from preprocessing.image_enhancement import LowLightEnhancer 
    from helper.mqtt import MQTTClientHandler
    import helper.utils
    import helper.config as config
except ImportError as e:
    print(f'Error import: {e}. Pastikan kernel berada di root direktori BlinkLink-Iot.')

# 3. Mocking SoundManager untuk bypass inisialisasi audio fisik
class DummySoundManager:
    def __init__(self, *args, **kwargs): pass
    def play_sound(self, *args, **kwargs): pass
    def stop_sound(self, *args, **kwargs): pass
    def play(self, *args, **kwargs): pass
    def play_voice(self, *args, **kwargs): pass
    def play_frequency(self, *args, **kwargs): pass

if hasattr(helper.utils, 'SoundManager'):
    helper.utils.SoundManager = DummySoundManager
    print('SoundManager berhasil di-mocking (Audio dinonaktifkan di Colab).')

# 4. Inisialisasi Model AI & Vision
print('Mempersiapkan model MediaPipe dan FaceNet... (Harap tunggu)')
face_detector = FaceMeshDetector()
face_auth = FaceAuthenticator(embeddings_path=config.FACE_EMBEDDINGS_PATH, tolerance=config.FACE_RECOGNITION_TOLERANCE)
blink_processor = BlinkProcessor(config)
enhancer = LowLightEnhancer()

# 5. Inisialisasi IoT / MQTT
print('Mengkoneksikan klien MQTT...')
mqtt_client = MQTTClientHandler(config.MQTT_BROKER, config.MQTT_PORT)
try:
    if mqtt_client.connect():
        mqtt_client.start_publisher_thread()
        print('MQTT Berhasil Terhubung.')
except Exception as e:
    print(f'Peringatan Koneksi MQTT: {e}')

print('Seluruh inisialisasi selesai!')


# 4. Image Processing (The Pipeline)
Loop utama Computer Vision untuk mendeteksi wajah, mengukur kedipan mata, dan menembak perintah MQTT ke ESP32.

In [ ]:
# CELL 3: Main Processing Loop (The Pipeline)
import time
import pygame
from IPython.display import display, Image

pygame.init()

# Hidupkan bridge kamera JavaScript
init_camera()
time.sleep(2) # Waktu warmup kamera lokal
print('Memulai pipeline Vision. Tekan Stop (Interrupt) di cell ini untuk mengakhiri.')

# Bikin Handle untuk In-Place Display agar UI kamera JS tidak terhapus (Menggantikan clear_output)
display_handle = display(Image(data=b''), display_id=True)

current_frame_count = 0
lamp_is_on = False
is_authorized = False
authorized_user = None
none_count = 0

try:
    while True:
        current_time = pygame.time.get_ticks()
        current_frame_count += 1
        
        # 1. Dapatkan Frame
        frame = get_frame()
        
        # PENCEGAHAN STUCK LOPING KOSONG
        if frame is None:
            none_count += 1
            if none_count % 30 == 0:
                print("Menunggu frame kamera... Pastikan Anda memberikan izin akses kamera (Allow).", flush=True)
            time.sleep(0.1) # Beri nafas pada CPU agar tidak crash
            continue
            
        none_count = 0 # Reset counter jika frame berhasil ditangkap
            
        # 2. Preprocessing / Low Light Enhancement
        is_dark = enhancer.is_low_light(frame)
        if is_dark:
            frame = enhancer.enhance(frame)

        # Status Variabel OSD
        auth_color = (0, 0, 255) # Merah default
        
        # 3. Eksekusi MediaPipe FaceMesh
        detected_faces = face_detector.detect(frame)
        found_authorized_user_this_frame = False

        for face_data in detected_faces:
            rect = face_data['rect']
            landmarks = face_data['landmarks']
            (x, y, w, h) = (rect.left(), rect.top(), rect.width(), rect.height())
            
            # 4. Keras FaceNet Authentication (Tiap 5 Frame)
            if current_frame_count % config.AUTH_CHECK_SKIP_FRAMES == 0:
                name, distance = face_auth.recognize_face(frame, rect)
                if name != "Unknown":
                    found_authorized_user_this_frame = True
                    is_authorized = True
                    authorized_user = name
            else:
                found_authorized_user_this_frame = True
            
            if is_authorized and authorized_user:
                auth_color = (0, 255, 0) # Hijau
                
                # 5. Deteksi Kedipan Wajah (Blink Processor)
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                blink_signal, blink_data, current_shape = blink_processor.process_frame(
                    gray, rect, None, current_time, current_frame_count, landmarks=landmarks
                )
                
                if blink_signal in ["TRIGGER_ACTION", "TRIGGER_MODE_SWITCH", "TRIGGER_SOS"]:
                    # 6. Kirim Action MQTT (Toggle Lampu)
                    lamp_is_on = not lamp_is_on
                    msg = "ON" if lamp_is_on else "OFF"
                    mqtt_client.publish_async(config.MQTT_TOPIC_LIGHT, msg)
                    cv2.putText(frame, 'ACTION TRIGGERED!', (20, 220), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

                # Render Progress Kedipan
                percentage = blink_data.get("progress_percentage", 0)
                if percentage > 0:
                    bar_x, bar_y = x, y + h + 30
                    bar_w = 150
                    progress_w = int(bar_w * percentage / 100)
                    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + progress_w, bar_y + 15), (0, 255, 0), -1)

            # Render Kotak Wajah dan Nama
            display_name = authorized_user if is_authorized and authorized_user else "Unknown"
            cv2.rectangle(frame, (x, y), (x+w, y+h), auth_color, 2)
            cv2.putText(frame, f"{display_name}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, auth_color, 2)

        # Render HUD Statis 
        lamp_status = "ON" if lamp_is_on else "OFF"
        cv2.putText(frame, f'Status Lampu: {lamp_status}', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 2)

        # 7. UPDATE Stream Video ke Layar Cell (Tanpa clear_output)
        # Encoding BGR (OpenCV) ke file bytes .jpg lalu di-update
        _, jpeg_img = cv2.imencode('.jpg', frame)
        display_handle.update(Image(data=jpeg_img.tobytes()))
        
except KeyboardInterrupt:
    print('\nPipeline dihentikan secara manual oleh user.')
except Exception as e:
    print(f'\nError runtime pada pipeline: {e}')
finally:
    try:
         mqtt_client.stop()
    except:
         pass
    print('Sistem telah offline dengan aman.')
